# LegPy tutorial II
### Photon beams

#### Fernando Arqueros, Jaime Rosado, Victor Moya


It is supposed that the user has already worked with the **LegPy tutorial I** and thus it is familiar with the construction of the objects and the MC parameters of LegPy. This tutorial is focused on the simulation of **photon beams**. However if the user activates the transportation of secondary electrons (e_transport = True) or the photon energy is over the pair creation threshold, it is advisable to read the tutorial for the simulation of electrons and positrons, **LegPy tutorial III**.  

Next it will be shown, using a number of examples, the procedures to visualize and analyze the results of the simulation of photon beams. In all these examples n_part is rather small in order to run the cases in a short time, thus leading to large statistical fluctuations. Obviously the user can enlarge n_part to get better statistics.  

### Previous steps

Again, as explained in **tutorial I**, if you have downloaded LegPy in you PC and are runnig this notebook with Jupyter or a similar application, ignore this step. But if you are using **Google Colab**, first you should clone the last version of the LegPy repository each time you inizialite Colab and add the directory where LegPy is located to the Python search path. Also you need an excel package. 

In [ ]:
#Only for Colab users!!
!git clone https://github.com/JaimeRosado/LegPy.git
import sys
sys.path.insert(0,'/content/LegPy')

# Colab users have to install also this package for managing excel files. 
!pip install xlsxwriter
import xlsxwriter

If not yet, the user has to import several modules as explained in **tutorial I**.

In [ ]:
import LegPy as lpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1) Photon beam on one medium

### 1.1) Display of photons tracks and/or interaction points

Useful for the visualization of the tracks of a few photons. The interaction points can be also displaied with a color scale (from blue to red) proportional to the logarithmic deposited energy (from low to high). Tracks of secondary electrons are not displayed.  

In [ ]:
medium = lpy.Medium(name='Sodium Iodide')
geometry = lpy.Geometry(name='cylinder', diam=50., z=20., n_x=30, n_y=30, n_z=100)
beam = lpy.Beam(name='parallel', theta=0.0, phi=0.0, diam=4., p_in=(0.,0.,0.))
spectrum = lpy.Spectrum(name='exponential', E_min=0.1, E_max=2., E_ch=0.3)
lpy.Plot_beam(medium, geometry, spectrum, beam, n_part=100, points=True) #tracks=True by default
#Alternative:
#result = lpy.MC(medium, geometry, spectrum, beam, n_part=100, tracks=True, points=False)

### 1.2) Angle and energy of outgoing photons

In [ ]:
medium = lpy.Medium(name='Water, Liquid')
geometry = lpy.Geometry(name='cylinder', diam=50., z=20., n_x=30, n_y=30, n_z=100)
beam = lpy.Beam(name='parallel', theta=0.0, phi=0.0)
spectrum = lpy.Spectrum(name='exponential', E_min=0.1, E_max=2., E_ch=.3)
result = lpy.MC(medium, geometry, spectrum, beam, n_part=1000, n_ang=10, n_E=50)

**Histograms of angular and energy distributions of outgoing photons**

Three histograms of the photons escaping the medium:
- Distribution of the zenith angle, $\frac{{\rm d}N}{{\rm d}\theta}$(rad$^{-1}$) vs $\theta$
- Energy spectrum, $\frac{{\rm d}N}{{\rm d}E}$(MeV$^{-1}$) vs E
- E(MeV) vs $\theta$(rad). Each 2D bin represents $\frac{dN}{d\theta dE}$(MeV$^{-1}$rad$^{-1}$)

Options acording to the parameter `interaction`: 
- True: Only photons that has suffered at least one interaction in the medium are included.
- False: Only primary photons leaving the medium
- 'both': All escaping photons 

Default is True

*Normalization*: For a monochromatic beam, the sum $\Sigma \frac{dN}{d\theta}\times d\theta$ represents, if `interaction = False` (`True`), the probability that an incoming photon escapes without (with) interacting in the medium. The same for the other spectra.

These histograms can be: 

- **plotted**

In [ ]:
result.plot_gamma_out_hists(interaction=True)

- **stored** in dataframes 

In [ ]:
ang_out_df = result.ang_out(interaction = True)
ang_out_df.head()

In [ ]:
E_out_df = result.E_out(interaction ='both')
E_out_df.head()

In [ ]:
E_ang_out_df = result.E_ang_out()
E_ang_out_df.head()

- **written** in excel files

In [ ]:
result.E_out_to_excel("E_out", interaction=True)

In [ ]:
result.ang_out_to_excel("ang_out", interaction='both')

In [ ]:
result.E_ang_out_to_excel("E_ang_out")

#### 1.2.1 Probabilities of interaction in the medium from these histograms

- Probability of no interaction

In [ ]:
E_out_df = result.E_out(interaction = False)
Ei = E_out_df[E_out_df.columns[0]]
dE = Ei[2] - Ei[1]
Pi = E_out_df[E_out_df.columns[1]]
P_ni = (Pi * dE).sum()
print('Fraction(%) of photons escaping the medium without interaction =', round(100. * P_ni, 2))

- Probability of escaping after, at least, one interaction. It also includes X-rays from photelectric absorption.

In [ ]:
E_out_df = result.E_out(interaction = True)
Ei = E_out_df[E_out_df.columns[0]]
dE = Ei[2] - Ei[1]
Pi = E_out_df[E_out_df.columns[1]]
P_i = (Pi * dE).sum()
print('Fraction(%) of photons leaving the medium after interacting =', round(100. * P_i, 2))

- Probability of absortion in the medium.

In [ ]:
P_a = 1. - P_ni - P_i
print('Fraction(%) of photons fully absorbed in the medium =', round(100. * P_a, 2))

The same result can be obtained with the dataframe `result.E_out(interaction = )`

#### 1.2.2 E (theta) of escaping photons

From the dataframe `result.E_ang_out` The energy spectrum of photons escaping the medium at given (zenith) angles can be easily calculated and plotted.

In [ ]:
plt.figure()
for theta in E_ang_out_df.columns[::3]:
    E_spect = E_ang_out_df.loc[:, theta]
    plt.scatter(E_spect.index, E_spect.values, label=str(round(theta, 2)))
plt.xlabel('E(MeV)', fontsize=15)
plt.ylabel('I(a.u.)', fontsize = 15) # por sr.
plt.yscale('log')
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.grid(True, which = 'both')
plt.legend(loc='upper right', fontsize = 12)
plt.title('Energy spectrum at several zenith angle (in rad.)')
plt.show()

### 1.3) Spectrum of the absorbed energy, Eab

The spectrum of the energy deposited in the medium, either directly by the photons (`e_transport = False`) or by the electrons (`e_transport = True`) is obtained with the methods of the object result with name **Eab**. This spectrum $\frac{{\rm d}N}{{\rm d}E}$(MeV$^{-1}$) vs $E$ can also be plotted, store in dataframes and written in excel files. See also the comments on 1.4)

*Normalization*: For a monochromatic beam (E$_0$), the sum $\Sigma$ $\frac{{\rm d}N}{{\rm d}E} \times {{\rm d}E}$ represents the probability that an incoming photon deposits some energy in the medium. The content of the last bin gives the probability of full energy deposition, $\frac{{\rm d}N}{{\rm d}E}$(E = E$_0$) $\times {{\rm d}E}$

In [ ]:
result.plot_Eab()

In [ ]:
Eab_df = result.Eab_to_df()
Eab_df.head()

In [ ]:
result.Eab_to_excel('Eab')

#### 1.3.1 Probability of energy deposition

In [ ]:
Ei = Eab_df[Eab_df.columns[0]]
dE = Ei[2] - Ei[1]
Pi = Eab_df[Eab_df.columns[1]]
P = (Pi * dE).sum()
print('fraction(%) of photons depositing some energy in the medium =', round(100. * P, 2))

This value in consistent with that obtained in 1.2.1. Note that photons that suffer coherent scattering leave the medium without depositing energy.

#### 1.3.2 Spectrum of energy deposition in a scintillator

- A point source of $^{60}$Co on NaI.

In [ ]:
medium = lpy.Medium(name='Sodium Iodide')
geometry = lpy.Geometry(name='cylinder', z = 5., diam = 5., n_z = 1, n_r = 1)
beam = lpy.Beam(name = 'isotropic', diam = 5., p_in = np.array ([0., 0., -2.])) # Point radiactive source 
spectrum = lpy.Spectrum('mono', E = 1.27) # of 60Co
result = lpy.MC(medium, geometry, spectrum, beam, n_part = 10000, n_ang=45, n_E=100, x_ray = True)
#
Eab_df = result.Eab_to_df()
dE = (Eab_df['Energy (MeV)'][2] - Eab_df['Energy (MeV)'][1]) * 1000. # deltaE (keV)
E = Eab_df['Energy (MeV)']*1000. # keV
P = Eab_df['MeV$^{-1}$'] / dE
#
plt.figure()
plt.step(E[1:], P[1:], where = 'mid',  linewidth = 2., c = 'r') # 
plt.xlabel('E(keV)', fontsize=15)
plt.ylabel('Probability (1/keV)', fontsize = 15) # por sr.
plt.yscale('log')
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.grid(True, which = 'both')
plt.title('NaI - $^{60}$Co', fontsize = 15)
plt.show()

Apart form the **photopeak** and the **Compton** profile, the escape of **X-rays** are observed (small peak close to the photopeak)

- The same case but with gamma rays with energy over the pair-creation threshold.

In [ ]:
medium = lpy.Medium(name='Sodium Iodide')
geometry = lpy.Geometry(name='cylinder', z = 5., diam = 5., n_z = 1, n_r = 1)
beam = lpy.Beam(name = 'isotropic', diam = 5., p_in = np.array ([0., 0., -2.])) # Point radiactive source 
spectrum = lpy.Spectrum('mono', E = 3.) # 
result = lpy.MC(medium, geometry, spectrum, beam, n_part = 100000, n_ang=45, n_E=100, x_ray = True, 
                e_transport = True)

Eab_df = result.Eab_to_df()
dE = (Eab_df['Energy (MeV)'][2] - Eab_df['Energy (MeV)'][1]) * 1000. # deltaE (keV)
E = Eab_df['Energy (MeV)']*1000. # keV
P = Eab_df['MeV$^{-1}$'] / dE
#
plt.figure()
plt.step(E[1:], P[1:], where = 'mid', linewidth = 2., c = 'r') # 
plt.xlabel('E(keV)', fontsize=15)
plt.ylabel('Probability (1/keV)', fontsize = 15) # por sr.
#plt.yscale('log')
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.grid(True, which = 'both')
plt.title('NaI - 3MeV photons', fontsize = 15)
plt.show()

The **escape peaks** of one and two **annihilation** photons is observed at ~2 and ~2.5 MeV. 

### 1.4) Kerma and Dose calculations

These dosimetric magnitudes can be obtained from the result methods named **Edep**. These methods provide the energy released by the involved particles (photons, electron/positrons) in keV/cm$^3$ units and the correspondong spatial distribution in the medium with the voxelization selected by the user, that is, cartesian Edep(x,y,z), cylindrical Edep(r,z) or spherical Edep(R). Note that **Edep** and **Eab** represent the same magnitude but **Eab** contains the energy spectrum while **Edep** the spatial distribution.

- If electron transportation is disabled (e_transport = False) the energy transferred to the electron is forced to be fully deposited in the interaction point and thus Edep represents in fact the energy **transferred** from photons to electrons. The user can easily compute the **Kerma** from Edep and the medium density. 

- If e_transport = True, Edep represents the **deposited** energy and therefore it allows the calculation of the **Dose**.

Note that at low photon energy, the tracks of the secondary electrons are very small. In addition Bremsstralung and pair creation are negligible (or forbidden). Therefore at low energy it is advisable to disable the transportation of electrons to save computing time, since Dose equals Kerma.

At high energy (several MeVs depending on medium), Bremsstralung (neglected in the current version) might be non-negligible and thus the Dose calculation might be affected since, in the real case, a fraction of the electrons energy is deposited far away of the electron tracks.

#### 1.4.1) Cylindral voxelization, Edep(r,z)

Only valid for a cylinder.

In [ ]:
medium = lpy.Medium(name='Tissue, Soft (ICRU Four-Component)')
geometry = lpy.Geometry(name='cylinder', diam=50., z=30., n_r=50, n_z=30)
beam = lpy.Beam(name='parallel', diam=10.)
spectrum = lpy.Spectrum('mono', E = 1.5)
result = lpy.MC(medium, geometry, spectrum, beam, n_part = 10000)

#### Spatial distribution can be:

- plotted

In [ ]:
result.plot_Edep()

- Downloaded to a 2D np.array

In [ ]:
Edep_np = result.Edep

 although it might be more convenient
 
- store in dataframes

In [ ]:
Edep_df = result.Edep_to_df()
Edep_df.head()

- written in excel file

In [ ]:
result.Edep_to_excel('Edep_cyl')

Note that above results are averaged over axial angle

- Total energy deposited in the medium can be calculated

In [ ]:
Vp = geometry.delta_v # np. array of pixel volumes for all r values (cm^3)
Ed = Edep_df.mul(Vp, axis=1).to_numpy().sum() # 
print('total energy deposit =', round(Ed, 3), 'keV')

#### 1.4.2) Cartesian voxelization, Edep(x,y,z)

Valid for any geometry.

In [ ]:
medium = lpy.Medium(name='Tissue, Soft (ICRU Four-Component)')
#geometry = lpy.Geometry(name='cylinder', diam=50., z=100., n_x=30, n_y=30, n_z=100)
geometry = lpy.Geometry(name='orthohedron', x = 30., y = 40., z = 50., n_x = 30, n_y=40, n_z=10)
beam = lpy.Beam(name='parallel', diam=10.)

spectrum = lpy.Spectrum('mono', E = 1.5)

spectrum = lpy.Spectrum(name='mono', E = 1.5)
result = lpy.MC(medium, geometry, spectrum, beam, n_part = 10000)

Spatial distribution can be:

- plotted

In [ ]:
result.plot_Edep()

- Downloaded to a 3D np.array.

In [ ]:
Edep_np = result.Edep

- stored in python binary file with extension .npy. This 3D energy distribution cannot be stored in an excel file or a dataframe, but can be saved in this file for later use.

In [ ]:
result.Edep_to_npy('Edep_cart')

- Total energy deposited in the medium can be calculated

In [ ]:
Vp = geometry.delta_v # pixel volume (cm^3)
Ed = result.Edep.sum() * Vp
print('total energy deposit =', round(Ed, 3), 'keV')

Personalized plots of the spatial distribution of Edep can be done with this tool:  

In [ ]:
z_ind = [0, 2, 5, 7, 9] # index of z layers.
prof_lev = [-5., -4., -3., -2., -1.] # dosis levels (adjust according the color bar)
result.plot_Edep_layers(axis="z", indexes=z_ind, c_profiles=True, lev=prof_lev)

In [ ]:
x_ind = [0, 9, 15, 20, 24] # index of x layers.
prof_lev =  [-5., -4., -3., -2., -1.] # dosis levels (adjust according the color bar)
result.plot_Edep_layers(axis="x", indexes=x_ind, c_profiles=True, lev=prof_lev)

#### 1.4.3) Spherical voxelization, Edep(R)

Only available if the photon source is located inside the sphere.

In [ ]:
medium = lpy.Medium(name='Tissue, Soft (ICRU Four-Component)')
geometry = lpy.Geometry(name='sphere', r = 15.0 , n_r = 15) # Spherical voxelization
beam = lpy.Beam(name = 'isotropic')
spectrum = lpy.Spectrum(name='mono', E = 1.5)
result = lpy.MC(medium, geometry, spectrum, beam, n_part = 10000)

#### Spatial distribution can be:

- plotted

In [ ]:
result.plot_Edep()

- Downloaded to a 1D np.array.

In [ ]:
Edep_np = result.Edep

- stored in dataframes

In [ ]:
Edep_df = result.Edep_to_df()
Edep_df.head()

- written in excel file

In [ ]:
result.Edep_to_excel('Edep_sph')

- Total energy deposited in the medium can be calculated

In [ ]:
Vp = geometry.delta_v # np. array of pixel volumes of r values (cm^3)
Ed = Edep_df.mul(Vp, axis=0).to_numpy().sum() # 
print('total energy deposit =', round(Ed, 3), 'keV')

#### 1.4.4) An example to calculate and compare Dose and Kerma 

In this example you can calculate and compare **dose** and **kerma** for a photon beam of 5 MeV crossing a water cylinder. After the dose build-up in the first milimeters kerma equals dose. 

***Note:*** Even a this energy (5 MeV) Bremsstralung is not relevant in water. 

In [ ]:
medium = lpy.Medium(name='Water, Liquid')
geometry = lpy.Geometry(name='cylinder', diam=50., z=20., n_r=1, n_z=50)
beam = lpy.Beam(name='parallel', diam=1.)
spectrum = lpy.Spectrum(name='mono', E=5.0)
result_d = lpy.MC(medium, geometry, spectrum, beam, e_transport=True, n_part = 10000)
result_k = lpy.MC(medium, geometry, spectrum, beam, e_transport=False, n_part = 10000)
#
Kerma_df = result_k.Edep_to_df()
depth_k = Kerma_df.index
Kerma = Kerma_df.values
#
Dose_df = result_d.Edep_to_df()
depth_d = Dose_df.index
Dose = Dose_df.values
#
plt.figure()
plt.scatter(depth_k, Kerma, color = 'r', label = 'Kerma', marker = 'o', s = 15.)
plt.scatter(depth_d, Dose, color = 'k',  label = 'Dose', marker = 'o', s = 15.)
plt.xlabel('depth(cm)', fontsize = 14)
plt.ylabel('Edep (keV/cm$^3$)', fontsize = 14)
plt.title('Dose and Kerma for a 5 MeV photon beam in water', fontsize = 14)
plt.grid(True, which = 'both')
plt.yscale('log')
plt.legend(loc='lower right', fontsize = 15)
plt.show()

### 1.5) Fluence

LegPy might record the **total** and **spectral fluence** of both primary and scattered photons in some locations of the media, as requested by the user. There are two options: fluence only along the z axis or fluence in the whole volume. Currently fluence can be only calculated in cylindrical and cartesian voxelizations.

#### 1.5.1) Fluence along the z axis

Total and spectral photon fluence is evaluated along the **z axis** in n_zloc+1 locations (from z = 0 to z = maximum depth) by counting photons traversing circular surfaces of the size of the voxels defined in 'geometry'. For both a cylinder with cylindrical voxelization and a sphere with spherical voxelization, the radius of these circular surfaces is diam/(2 * n_r). For an object with cartesian voxelization, the radius is the minimum of x/(2 * n_x) and y/(2 * n_y). The fluence is calculated correcting for the direction of each photon. In this way it is calculated for all these z locations the number of photons per unit area perpendicular to the surface.

Fluence results are given **per incident photon** beam. Total fluence in units of m$^{-2}$ and spectral fluence in MeV$^{-1}$m$^{-2}$

In [ ]:
medium = lpy.Medium(name='Water, Liquid')
geometry = lpy.Geometry(name='cylinder', diam=50., z=100., n_r=25, n_z=50)
beam = lpy.Beam(name='parallel', diam=50)
spectrum = lpy.Spectrum(name='mono', E = 1.)
result = lpy.MC(medium, geometry, spectrum, beam, n_part = 10000, fluence = 'z', n_zloc=10)

- Fluence plot

    - First histogram: total fluence as function of depth
    - Second histogram: spectral fluence at the entrance (z=0)
    - Third histogram: spectral fluence at the exit (maximum z)

In [ ]:
result.plot_fluence()

- Fluence data can be stored in a dataframe or in an excel file

In [ ]:
fluence_df = result.fluence_to_df()
fluence_df.head(12)

- written in excel file

In [ ]:
result.fluence_to_excel("fluence_z")

#### 1.5.2) Fluence in the whole volume 

If requested by the user, the surfaces used to measured the fluence can be extended to the whole medium volume.   

#### cylindrical voxelization

Only valid for a cylinder.

In [ ]:
r = 5.
nr = 5
medium = lpy.Medium(name='Water, Liquid')
geometry = lpy.Geometry(name='cylinder', diam=2.*r, z=50., n_r=nr, n_z=50)
beam = lpy.Beam(name='parallel')
spectrum = lpy.Spectrum(name='mono', E=0.3)
result = lpy.MC(medium, geometry, spectrum, beam, n_part = 10000, fluence = True)

Same of above but total and spectral photon fluence is evaluated on surfaces perpendicular to the z axis in the (`n_z` + 1, `n_r`) locations given by the voxelization defined in `Geometry`. For each `n_z` + 1 locations (0., `z/n_z`, `2z/n_z`, ..., `z`) these surfaces are `n_r` rings of width equals `diam / (2 * n_r)`

- Fluence plot: similar to that of the previous example. The user has to select the radial bin `ri`.

In [ ]:
ri = 2 # any value smaller than nr 
result.plot_fluence(ri=ri)
r_bin = r/nr * ri + 1./2.*r/nr # mean value (cm) of radial bin i 
print('radial bin at', round(r_bin, 2), 'cm' )

- Fluence data can be stored in a dataframe or in an excel file

In [ ]:
fluence_df = result.fluence_to_df(ri=ri)
fluence_df.head()

- written in excel file

In [ ]:
result.fluence_to_excel("3D_fluence_cyl", ri=ri)

#### cartesian voxelization

Valid for any medium geometry

In [ ]:
x = 50.
nx = 5
y = 40.
ny = 2
medium = lpy.Medium(name='Water, Liquid')
geometry = lpy.Geometry(name='orthohedron', x=x, y=y, z=50., n_x=nx, n_y=ny, n_z=50)
beam = lpy.Beam(name='parallel')
spectrum = lpy.Spectrum(name='mono', E=0.3)
result = lpy.MC(medium, geometry, spectrum, beam, n_part = 10000, fluence = True)

Same of above but total and spectral fluence are evaluated on surfaces perpendicular to the z axis in the (`n_x`, `n_y`, `n_z`) locations given by the voxelization defined in `Geometry`, that is, for each z value, fluence is evaluated in `n_x` * `n_y` rectangles of size (`x/n_x`, `y/n_y`).

- Fluence plot: similar to that of the previous example. The user has to select the cartesian bin `xi`,`yi` 

In [ ]:
xi = 0 # any value smaller than nx
yi = 1 # any value smaller than ny
result.plot_fluence(xi=xi, yi = yi)
x_bin = x/nx * xi + 1./2.*x/nx # mean value (cm) of x bin i
y_bin = y/ny * yi + 1./2.*y/ny # mean value (cm) of y bin i
print('(x,y) bin = ','(',round(x_bin, 2), ',', round(y_bin, 2),')',  'cm' )

- Fluence data can be stored in a dataframe or in an excel file

In [ ]:
fluence_df = result.fluence_to_df(xi=xi, yi=yi)
fluence_df.head()

- written in excel file

In [ ]:
result.fluence_to_excel("3D_fluence_cart", xi=xi, yi=yi)

#### 1.5.3) From fluence to Dose

The simulation of the spectral photon fluence, $\frac{{\rm d}\Phi}{{\rm d}E}$, is very useful because it allows the calculation of the dose using the tabulated values of the absorption coefficients $\mu_{ab}$. At a given point in the medium, the dose, $D$ can be calculated as

$D = \int E \frac{{\rm d}\Phi}{{\rm d}E}(E)\cdot \mu_{ab}(E) \cdot{\rm d} E$

The number of photons crossing the reference surfaces is much higher that the number of interactions in the corresponding voxels and thus the statistical fluctuations of the fluence simulation are much smaller than those of **Edep**. The current version of LegPy includes tables of the absorption coefficients of all media.

***Note:*** The effect of Bremsstralung is partly included in this method since $\mu_{ab}$ is the fraction of the photon energy locally deposited per unit mass depth traversed, and thus discounting radiative losses.  

Next it will be shown a simple example of the dose calculation for the simulated fluence.

## 2) Simulations on two-media geometries

The simulation on geometries with two media can be performed in a similar way but you have to:

- Define (construct) both media.
- Give the position of the interface surface in your geometry.

Note that a common voxelization in used in both media. The result can be analyze with the same tools described  above. Next you can find some examples.

### 2.1) A divergent photon beam on a two-media cylinder ###

In [ ]:
water = lpy.Medium(name='Water, Liquid') # NIST
gold = lpy.Medium(name='Au') # NIST
E = 1. # MeV
spectrum = lpy.Spectrum(name='mono', E = E)
beam = lpy.Beam(name = 'isotropic', diam = 2.0, p_in = np.array ([0., 0., -5]))

#### 2.1.1) z interface   ####

Interface: horizontal plane at `z = z_int`; water in `[0, z_int]` and gold in `[z_int, z]`.

In [ ]:
geometry = lpy.Geometry(name='cylinder', z=2., r=2., n_z=20, n_r=20, z_int=1.5)
result = lpy.MC([water, gold], geometry, spectrum, beam, n_part=10000)
result.plot_Edep()

#### 2.1.2) r interface   ####

Interface: Inner cylinder of radius equals `r_int`; water in `[0, r_int]` and gold in `[r_int, r]`.

In [ ]:
geometry = lpy.Geometry(name='cylinder', z=2., r=2., n_z=20, n_r=20, r_int=1.)
result = lpy.MC([water, gold], geometry, spectrum, beam, n_part=10000)
result.plot_Edep()

### 2.2) An isotropic photon-source in the center of a two media sphere. ###

In [ ]:
water = lpy.Medium(name='Water, Liquid')
gold = lpy.Medium(name='Au')
E = 1.
beam = lpy.Beam(name='isotropic', p_in=np.array([0., 0., 0.]))

#### 2.2.1) r interface   ####

Interface: inner sphere of radius `r_int`; medium_1 in the sphere `[0, r_int]` and medium_2 in `[r_int, r]`.

In [ ]:
geometry = lpy.Geometry(name='sphere', r=5., n_r=50, r_int=3.)
result = lpy.MC([water, gold], geometry, spectrum, beam, n_part=10000)
result.plot_Edep()

#### 3.3.2) z interface   ####

Interface: horizontal plane at `z = z_int`; water in `-r > z > z_int` and gold in `z_int > z > r`.

In [ ]:
geometry = lpy.Geometry(name='sphere', diam=10., n_r=50, n_z=10, z_int=2.5)
result = lpy.MC([water, gold], geometry, spectrum, beam, n_part=10000)
result.plot_Edep()